In [31]:
import os
import glob
import cv2
import numpy as np
import mediapipe as mp
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [32]:
dataset_dir = "./asl_dataset"

In [33]:
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
)

In [34]:
def extract_landmarks(img):
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    if not results.multi_hand_landmarks:
        return None

    hand = results.multi_hand_landmarks[0]

    landmarks = []

    for lm in hand.landmark:
        landmarks.extend([lm.x, lm.y, lm.z])

    landmarks = np.array(landmarks)

    # Normalize landmarks
    landmarks = landmarks - landmarks.mean()
    landmarks = landmarks / (landmarks.std() + 1e-6)

    return landmarks


In [35]:
def augment_image(img):
    augmented = []

    h, w = img.shape[:2]

    # Rotate left
    M1 = cv2.getRotationMatrix2D((w/2, h/2), -10, 1)
    rot1 = cv2.warpAffine(img, M1, (w, h))
    augmented.append(rot1)

    # Rotate right
    M2 = cv2.getRotationMatrix2D((w/2, h/2), 10, 1)
    rot2 = cv2.warpAffine(img, M2, (w, h))
    augmented.append(rot2)

    # Brightness
    bright = cv2.convertScaleAbs(img, alpha=1.1, beta=15)
    augmented.append(bright)

    return augmented


In [36]:
X = []
y = []

for label in sorted(os.listdir(dataset_dir)):
    folder = os.path.join(dataset_dir, label)

    if not os.path.isdir(folder):
        continue

    for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
        for fp in glob.glob(os.path.join(folder, ext)):

            img = cv2.imread(fp)

            if img is None:
                continue

            feat = extract_landmarks(img)

            if feat is None:
                continue

            X.append(feat)
            y.append(label)

X = np.array(X)
y = np.array(y)

print("Original dataset:", X.shape)

Original dataset: (9509, 63)


In [37]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

In [38]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=42,
    shuffle=True
)


In [39]:
aug_X = []
aug_y = []

for label in sorted(os.listdir(dataset_dir)):
    folder = os.path.join(dataset_dir, label)

    if not os.path.isdir(folder):
        continue

    for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
        for fp in glob.glob(os.path.join(folder, ext)):

            img = cv2.imread(fp)

            if img is None:
                continue

            augmented_imgs = augment_image(img)

            for aug in augmented_imgs:
                feat = extract_landmarks(aug)

                if feat is None:
                    continue

                aug_X.append(feat)
                aug_y.append(label)

aug_X = np.array(aug_X)
aug_y = le.transform(np.array(aug_y))

In [40]:
X_train = np.concatenate([X_train, aug_X])
y_train = np.concatenate([y_train, aug_y])

print("Training set after augmentation:", X_train.shape)

Training set after augmentation: (36500, 63)


In [41]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

In [42]:
param_grid = {
    'svm__C': [0.1, 1, 10],
    'svm__gamma': [0.001, 0.01, 'scale'],
    'svm__kernel': ['rbf']
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


,estimator,"Pipeline(step...svm', SVC())])"
,param_grid,"{'svm__C': [0.1, 1, ...], 'svm__gamma': [0.001, 0.01, ...], 'svm__kernel': ['rbf']}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [43]:
best_model = grid.best_estimator_

In [44]:
print("Best Params:", grid.best_params_)

train_acc = best_model.score(X_train, y_train)
test_acc = best_model.score(X_test, y_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

pred = best_model.predict(X_test)

print(classification_report(
    y_test,
    pred,
    target_names=le.classes_
))

print("Final Accuracy:", accuracy_score(y_test, pred))

Best Params: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Train Accuracy: 0.9987945205479452
Test Accuracy: 0.9968454258675079
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           1       1.00      1.00      1.00        13
           2       1.00      1.00      1.00        11
           3       1.00      1.00      1.00        15
           4       1.00      1.00      1.00        15
           5       1.00      1.00      1.00        15
           6       0.86      1.00      0.92        12
           7       0.93      0.93      0.93        14
           8       0.92      0.92      0.92        13
           9       1.00      1.00      1.00        15
           a       0.98      1.00      0.99        64
           b       1.00      1.00      1.00        74
           c       1.00      1.00      1.00        67
           d       1.00      1.00      1.00        74
           e       1.00      1.00      1.00    

In [45]:
joblib.dump({
    "model": best_model,
    "label_encoder": le
}, "svm_asl_mp_model.joblib")

print("Model saved successfully")

Model saved successfully
